In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from datetime import datetime

import warnings

warnings.filterwarnings("ignore")

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"
MODELS_DIR = PROJECT_ROOT / "MODELS"

FEATURE_FILE = PROCESSED_DIR / "cars_features.csv"
MODEL_PATH = MODELS_DIR / "car_price_model.joblib"

In [3]:
model = joblib.load(MODEL_PATH)

print("Car price model loaded successfully.")

Car price model loaded successfully.


In [4]:
FEATURE_FILE = (
    PROJECT_ROOT /
    "DATA" /
    "PROCESSED" /
    "cars_features.csv"
)

training_data = pd.read_csv(
    FEATURE_FILE
)

TARGET_CANDIDATES = [
    "selling_price",
    "price",
    "resale_price"
]

TARGET = next(
    (
        col for col in TARGET_CANDIDATES
        if col in training_data.columns
    ),
    None
)

FEATURE_COLUMNS = [
    col for col in training_data.columns
    if col != TARGET
]

print("Target:", TARGET)

print("\nModel features:")

for feature in FEATURE_COLUMNS:
    print("-", feature)

Target: price

Model features:
- id
- year
- brand
- full_model_name
- model_name
- distance_travelled(kms)
- fuel_type
- city
- car_age


In [5]:
# Start with median values for numeric features
# and most frequent values for categorical features.

test_input = {}

for column in FEATURE_COLUMNS:

    if pd.api.types.is_numeric_dtype(
        training_data[column]
    ):
        test_input[column] = (
            training_data[column]
            .median()
        )

    else:
        mode = training_data[column].mode()

        if len(mode) > 0:
            test_input[column] = mode.iloc[0]
        else:
            test_input[column] = "Unknown"

test_car = pd.DataFrame([test_input])

test_car

,id,year,brand,full_model_name,model_name,distance_travelled(kms),fuel_type,city,car_age
0,862.0,2016.0,Hyundai,Ford EcoSport Titanium 1.5L TDCi,Creta,49000.0,Diesel,Chennai,10.0


In [6]:
CURRENT_YEAR = datetime.now().year

if "year" in test_car.columns:
    test_car["year"] = 2019

if "car_age" in test_car.columns:
    test_car["car_age"] = CURRENT_YEAR - 2019

if "km_driven" in test_car.columns:
    test_car["km_driven"] = 50000

if "kilometers_driven" in test_car.columns:
    test_car["kilometers_driven"] = 50000

print("Test car:")
display(test_car)

Test car:


,id,year,brand,full_model_name,model_name,distance_travelled(kms),fuel_type,city,car_age
0,862.0,2019,Hyundai,Ford EcoSport Titanium 1.5L TDCi,Creta,49000.0,Diesel,Chennai,7


In [7]:
prediction = model.predict(
    test_car
)

predicted_price = prediction[0]

print(
    f"Predicted Car Price: ₹{predicted_price:,.2f}"
)

Predicted Car Price: ₹1,248,670.00


In [8]:
def predict_car_price(car_data):
    """
    Predict the market price of a car.
    """

    input_df = pd.DataFrame(
        [car_data]
    )

    prediction = model.predict(
        input_df
    )

    return float(prediction[0])

In [9]:
result = predict_car_price(
    test_input
)

print(
    f"Predicted Price: ₹{result:,.2f}"
)

Predicted Price: ₹1,029,253.33


In [10]:
test_cases = []

for multiplier in [0.8, 1.0, 1.2]:

    case = test_input.copy()

    if "km_driven" in case:
        case["km_driven"] = (
            50000 * multiplier
        )

    if "kilometers_driven" in case:
        case["kilometers_driven"] = (
            50000 * multiplier
        )

    test_cases.append(case)

In [11]:
for i, case in enumerate(
    test_cases,
    start=1
):

    prediction = predict_car_price(
        case
    )

    print(
        f"Test Case {i}: "
        f"₹{prediction:,.2f}"
    )

Test Case 1: ₹1,029,253.33
Test Case 2: ₹1,029,253.33
Test Case 3: ₹1,029,253.33


In [12]:
prediction_output = {
    "predicted_price": round(
        float(predicted_price),
        2
    ),
    "currency": "INR"
}

prediction_output

{'predicted_price': 1248670.0, 'currency': 'INR'}